# Speeding Up pandas on Real Estate Sales (DVF)

This notebook applies the techniques of the "Speeding Up Computations" chapter to a real, large dataset: France's real estate sales (DVF), 10.8 million sales from 2014 to 2022. Each part measures what it compares, so that you see the gain:

1. Memory and loading: measure, read less, choose dtypes
2. Loops and vectorization: the distance from each sale to Nantes, then its zone
3. Fast grouping, readable filtering: `agg` rather than `apply`, `query` and `eval`
4. Compiling with Numba: `@njit`, `prange`, a loop that doesn't vectorize, `engine="numba"`
5. Scaling up: Modin, Polars and DuckDB on the nine years

Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed. The cells after a solution print the checks and the timings: compare your results with them. The exercises marked "optional" aren't needed afterwards: skip them if time runs short.

The notebook runs without a GPU, on the 2 cores of Colab or of the Codespace, in a few minutes. The timings the exercises give are orders of magnitude, measured on 2 cores: yours will vary, the ratios between methods much less.

## Setup

The first cell downloads DVF from shuuchuu/datasets (635 MB, one Parquet folder per year) and installs Modin, which Colab lacks. The second imports the libraries and defines three measuring tools:

- `measure(function, repeats)`: the best time, in seconds, of several calls of `function()`, like `%timeit`, but returning the number;
- `stopwatch(function)`: the result of a single, long call, and its time;
- `time_table(times)`: a dictionary of timings as a table, from the slowest to the fastest, with the gain over the slowest.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!pip install -q "modin[dask]"

In [ ]:
import time
import timeit

import duckdb
import numpy as np
import pandas as pd
import polars as pl
from geopy.distance import geodesic
from numba import njit, prange

PATH = "dvf/dvf-linearized-2014-2022.parquet"
YEAR_2022 = [("annee_mutation", "==", 2022)]


def measure(function, repeats=3):
  """The best time, in seconds, of `repeats` calls of `function()`."""
  return min(timeit.repeat(function, number=1, repeat=repeats))


def stopwatch(function):
  """The result of one call of `function()`, and its time in seconds."""
  start = time.perf_counter()
  result = function()
  return result, time.perf_counter() - start


def time_table(times):
  """The timings (name → seconds), from the slowest to the fastest, and their gain."""
  seconds = pd.Series(times).sort_values(ascending=False)
  return pd.DataFrame(
    {
      "seconds": seconds.round(4),
      "× faster than the slowest": (seconds.max() / seconds).round(1),
    }
  )

## 1. Memory and loading

DVF is a partitioned Parquet dataset: one folder per year (`annee_mutation=2014/`…), which `pd.read_parquet` reads as one table. Parts 1 to 4 work on the year 2022 (508,837 sales, January to June), which `filters=` reads alone; part 5, on the nine years.

In [ ]:
%time dvf = pd.read_parquet(PATH, filters=YEAR_2022)
dvf.shape

### 1.1 Measuring memory (optional)

How much memory does `dvf` take? With `memory_usage(deep=True)`, print the table's total memory in MB (10⁶ bytes), then its five heaviest columns, with their dtype (`dvf.dtypes`). Compare with the total of `memory_usage()` without `deep=True`: where does the gap come from?

You should find about 206 MB, 155 of them for four string columns and the `id_mutation` index, against 72 MB without `deep=True`.

In [ ]:
# Your code here

#### Additional hints

- `dvf.memory_usage(deep=True)` returns a `Series`: the number of bytes of each column, plus an `Index` row; its sum divided by `1e6` gives MB.
- `.nlargest(5)` keeps its five largest values; `dvf.dtypes.reindex(...)` gives these columns' dtypes (`NaN` for the `Index`, which isn't one).
- Without `deep=True`, an `object` column counts 8 bytes per row: a pointer to each string, not the string itself.

A skeleton, where each `...` is yours to fill in:

```python
memory = dvf.memory_usage(deep=True)
print(f"With deep=True: {memory.sum() / 1e6:.0f} MB")
shallow = ...
print(f"Without deep=True: {shallow / 1e6:.0f} MB")
heaviest = memory.nlargest(5)
pd.DataFrame({"MB": (heaviest / 1e6).round(1), "dtype": ...})
```

#### Solution

In [ ]:
memory = dvf.memory_usage(deep=True)
print(f"With deep=True: {memory.sum() / 1e6:.0f} MB")
shallow = dvf.memory_usage().sum()
print(f"Without deep=True: {shallow / 1e6:.0f} MB")
# object strings: one Python object per value, which only deep=True counts.
heaviest = memory.nlargest(5)
pd.DataFrame(
  {"MB": (heaviest / 1e6).round(1), "dtype": dvf.dtypes.reindex(heaviest.index)}
)

### 1.2 Reading less

The rest of the notebook only uses the twelve columns of `COLUMNS` (cell below). Read 2022 again with `columns=COLUMNS` on top of `filters=`, into a `dvf` variable that replaces the previous one, and time the read with `%time`: compare it with that of the whole table. Print the new table's memory.

The `id_mutation` index, made of strings, is read even with `columns=`. Remove it with `reset_index(drop=True)`, which replaces it with a `RangeIndex` (integers computed on demand, with no memory), and print the memory again.

You should find a read about twice as fast, 85 MB, then 55 MB without the index.

In [ ]:
COLUMNS = [
  "mois_mutation",
  "jour_mutation",
  "nature_mutation",
  "valeur_fonciere",
  "code_postal",
  "nom_commune",
  "code_departement",
  "surface_reelle_bati_total",
  "nombre_maisons",
  "nombre_appartements",
  "latitude",
  "longitude",
]

In [ ]:
# Your code here

#### Additional hints

- `%time dvf = ...` times an assignment and keeps its result.
- `filters=YEAR_2022` as above; `columns=` takes a list of column names.
- `drop=True` throws the index away instead of making it a column.

A skeleton, where each `...` is yours to fill in:

```python
%time dvf = pd.read_parquet(PATH, columns=..., filters=...)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} MB")
dvf = ...
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} MB without the index")
```

#### Solution

In [ ]:
%time dvf = pd.read_parquet(PATH, columns=COLUMNS, filters=YEAR_2022)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} MB")
dvf = dvf.reset_index(drop=True)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} MB without the index")

### 1.3 Choosing dtypes, column by column

Rather than converting the columns one by one, write `reduce_dtypes(df)`, which loops over the columns of `df` with `df.items()` (pairs of name and `Series`) and returns a copy of `df` with reduced dtypes:

- an `object` column with less than 10% distinct values becomes `category`;
- an integer column gets the smallest dtype that keeps all its values, with `pd.to_numeric(..., downcast=...)`: `"unsigned"` if it has no negative value, `"integer"` otherwise.

The other columns stay as they are: `nature_mutation` and `code_departement` already are `category` (Parquet stores them as dictionaries, and pandas reads them back that way), and a floating-point number can't be reduced without losing precision. The cell after the solution checks that `reduce_dtypes(dvf)` keeps the same values, and prints its memory.

You should find `nom_commune` as `category`, `code_postal` as `uint32`, `nombre_maisons` and `nombre_appartements` as `uint8`, and less than 25 MB in all: more than eight times less than the 206 MB of the table you started from.

**Useful documentation**

- https://pandas.pydata.org/docs/reference/api/pandas.api.types.is_integer_dtype.html

In [ ]:
def reduce_dtypes(df):
  """A copy of `df` with reduced dtypes."""
  ...  # Your code here

#### Additional hints

- `for name, column in df.items():` goes over the columns; `column.nunique()` counts its distinct values, to compare with `0.1 * len(column)`.
- `column.dtype == "object"` spots the strings, `pd.api.types.is_integer_dtype(column)` the integers, signed or not.
- Change a copy, `reduced = df.copy()`, to leave `df` untouched while you loop over it.
- An example to check: `reduce_dtypes(pd.DataFrame({"n": range(20), "city": ["Lyon"] * 20})).dtypes` gives `uint8` for `n` and `category` for `city`.

A skeleton, where each `...` is yours to fill in:

```python
def reduce_dtypes(df):
  reduced = df.copy()
  for name, column in df.items():
    if column.dtype == "object" and ...:
      reduced[name] = ...
    elif pd.api.types.is_integer_dtype(column):
      downcast = "unsigned" if ... else "integer"
      reduced[name] = pd.to_numeric(column, downcast=downcast)
  return reduced
```

#### Solution

In [ ]:
def reduce_dtypes(df):
  """A copy of `df` with reduced dtypes."""
  reduced = df.copy()
  for name, column in df.items():
    if column.dtype == "object" and column.nunique() < 0.1 * len(column):
      reduced[name] = column.astype("category")
    elif pd.api.types.is_integer_dtype(column):
      downcast = "unsigned" if column.min() >= 0 else "integer"
      reduced[name] = pd.to_numeric(column, downcast=downcast)
  return reduced

In [ ]:
print(f"Before: {dvf.memory_usage(deep=True).sum() / 1e6:.0f} MB")
reduced = reduce_dtypes(dvf)
print(f"After: {reduced.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print("Same values:", reduced.astype(object).equals(dvf.astype(object)))

The table as the rest of the notebook uses it, with the dtypes `reduce_dtypes` reduced, and its memory:

In [ ]:
dvf = reduce_dtypes(dvf)
dvf.info(memory_usage="deep")

## 2. Loops and vectorization

How far is each sale from the center of Nantes? `geopy.distance.geodesic` computes it exactly, on the Earth's ellipsoid, but one point at a time, in Python: about a tenth of a millisecond per point. The next cell uses it as a reference on 2,000 sales of Loire-Atlantique.

In [ ]:
%%time
NANTES = 47.218371, -1.553621  # latitude, longitude of the center of Nantes
loire = dvf[dvf["code_departement"] == "44"]  # Loire-Atlantique
sample = loire.head(2000)
reference = np.array(
  [
    geodesic(NANTES, (lat, lon)).km
    for lat, lon in zip(sample["latitude"], sample["longitude"])
  ]
)

### 2.1 The haversine formula in NumPy

On a sphere of radius $R$ = 6,371 km, the haversine formula gives the distance between two points of latitudes $\varphi_1$, $\varphi_2$ and longitudes $\lambda_1$, $\lambda_2$, in radians:

$$d = 2R \arcsin \sqrt{\sin^2 \frac{\varphi_2 - \varphi_1}{2} + \cos \varphi_1 \cos \varphi_2 \sin^2 \frac{\lambda_2 - \lambda_1}{2}}$$

Write `haversine(lat1, lon1, lat2, lon2)`, which takes degrees and returns km, with NumPy's functions only (`np.radians`, `np.sin`, `np.cos`, `np.arcsin`, `np.sqrt`): it then works on two points as well as on whole columns, with no loop. The cell after the solution compares it with `geodesic` on the 2,000 sales of `sample`.

You should find `haversine(48.8566, 2.3522, *NANTES)` ≈ 343 km (Paris–Nantes), and a relative gap to `geodesic` under 0.5%: the Earth isn't quite a sphere.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.arcsin.html

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
  """The distance in km between (lat1, lon1) and (lat2, lon2), given in degrees."""
  ...  # Your code here

#### Additional hints

- First convert the four angles to radians, one `np.radians` per variable.
- $\sin^2 x$ is written `np.sin(x) ** 2`.
- With no `if` and no loop, the same function accepts numbers, NumPy arrays and `Series`: NumPy's operations apply element by element, with broadcasting between one point and a column.

A skeleton, where each `...` is yours to fill in:

```python
def haversine(lat1, lon1, lat2, lon2):
  lat1, lon1 = np.radians(lat1), np.radians(lon1)
  lat2, lon2 = ...
  a = np.sin((lat2 - lat1) / 2) ** 2 + ...
  return 2 * 6371 * ...
```

#### Solution

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
  """The distance in km between (lat1, lon1) and (lat2, lon2), given in degrees."""
  lat1, lon1 = np.radians(lat1), np.radians(lon1)
  lat2, lon2 = np.radians(lat2), np.radians(lon2)
  a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
  )
  return 2 * 6371 * np.arcsin(np.sqrt(a))

In [ ]:
print(f"Paris–Nantes: {haversine(48.8566, 2.3522, *NANTES):.0f} km")
distances = haversine(*NANTES, sample["latitude"], sample["longitude"])
gap = np.abs(distances - reference) / reference
print(f"Largest relative gap to geodesic: {gap.max():.2%}")

### 2.2 Five loops, and a condition with no loop

The cell below computes the distance to Nantes of each sale of `loire` (18,260 sales) by calling `haversine` on one sale at a time, in five ways (`iterrows`, `apply(axis=1)`, `itertuples`, `zip`, `np.vectorize`), then with a single call on the whole columns, and shows their timings.

In [ ]:
times = {}
times["iterrows"] = measure(
  lambda: [
    haversine(*NANTES, sale["latitude"], sale["longitude"])
    for _, sale in loire.iterrows()
  ],
  repeats=1,
)
times["apply(axis=1)"] = measure(
  lambda: loire.apply(
    lambda sale: haversine(*NANTES, sale["latitude"], sale["longitude"]), axis=1
  ),
  repeats=1,
)
times["itertuples"] = measure(
  lambda: [
    haversine(*NANTES, sale.latitude, sale.longitude) for sale in loire.itertuples()
  ],
  repeats=1,
)
times["zip"] = measure(
  lambda: [
    haversine(*NANTES, lat, lon)
    for lat, lon in zip(loire["latitude"], loire["longitude"])
  ],
  repeats=1,
)
times["np.vectorize"] = measure(
  lambda: np.vectorize(haversine)(*NANTES, loire["latitude"], loire["longitude"]),
  repeats=1,
)
times["vectorized"] = measure(
  lambda: haversine(*NANTES, loire["latitude"], loire["longitude"])
)
time_table(times)

The vectorized version is tens to hundreds of times faster than the best loop. The five loops call `haversine` once per sale, in Python, with numbers: each call costs as much as the vectorized computation on thousands of sales. `iterrows` and `apply(axis=1)` also build one `Series` per row, and `np.vectorize` is only a more convenient Python loop.

A condition vectorizes too. Classify each 2022 sale by its distance to Nantes: "Nantes" under 10 km, "agglomération" under 30 km, "reste" (the rest) beyond. First compute `distances`, the distance to Nantes of every sale of `dvf`, with a single call of `haversine`. Then classify them in two ways, and store their times in the `zone_times` dictionary, under the names `"apply"` and `"np.select"`:

- a `zone(distance)` function, with `if`, applied to each distance by `distances.apply(zone)` (a single repeat);
- `np.select`, with no Python loop.

The cell after the solution checks that both give the same zones, then prints the number of sales in each zone and the timings.

You should find about 6,350 sales in the "Nantes" zone and 4,270 in the "agglomération" one, and `np.select` about twenty times faster than `apply`, or more.

In [ ]:
zone_times = {}
# Your code here

#### Additional hints

- `zone` takes a number and returns a string: `zone(5)` is `"Nantes"`, `zone(25)` is `"agglomération"` and `zone(100)` is `"reste"`.
- `np.select(conditions, choices, default=...)` takes a list of masks and a list of values of the same length: each element gets the value of the **first** true condition, `default` otherwise. After `distances < 10`, the `distances < 30` condition thus only catches the sales between 10 and 30 km.
- `np.select` is timed inside a `lambda`, as `apply` is: `measure(lambda: np.select(...))`.

A skeleton, where each `...` is yours to fill in:

```python
distances = haversine(*NANTES, dvf["latitude"], dvf["longitude"])


def zone(distance):
  if distance < 10:
    return "Nantes"
  ...


zone_times = {}
zone_times["apply"] = measure(lambda: distances.apply(zone), repeats=1)
zone_times["np.select"] = measure(
  lambda: np.select([distances < 10, ...], [...], default=...)
)
```

#### Solution

In [ ]:
distances = haversine(*NANTES, dvf["latitude"], dvf["longitude"])


def zone(distance):
  if distance < 10:
    return "Nantes"
  if distance < 30:
    return "agglomération"
  return "reste"


zone_times = {}
zone_times["apply"] = measure(lambda: distances.apply(zone), repeats=1)
zone_times["np.select"] = measure(
  lambda: np.select(
    [distances < 10, distances < 30], ["Nantes", "agglomération"], default="reste"
  )
)

In [ ]:
zones = np.select(
  [distances < 10, distances < 30], ["Nantes", "agglomération"], default="reste"
)
print("Same zones:", (distances.apply(zone).to_numpy() == zones).all())
print(pd.Series(zones).value_counts())
time_table(zone_times)

## 3. Fast grouping, readable filtering

A town's price per m²: the sum of the property values of its sales of buildings, divided by the sum of their areas. A town is identified by its department and its name, since towns of different departments share names. The naive version passes each group to a Python function with `apply`; it takes several seconds:

In [ ]:
built = dvf[(dvf["valeur_fonciere"] > 0) & (dvf["surface_reelle_bati_total"] > 0)]
towns = built.groupby(["code_departement", "nom_commune"], observed=True)


def town_price_m2(sales):
  return sales["valeur_fonciere"].sum() / sales["surface_reelle_bati_total"].sum()


%time naive = towns.apply(town_price_m2, include_groups=False)
naive.head()

### 3.1 Aggregate, then divide

Compute the same price per m² without `apply`. A named aggregation of `towns` (`agg`) gives at once, per town, the sum of the values (column `value`), that of the areas (`area`) and the number of sales (`sales`), in a `totals` DataFrame; dividing two of its columns gives the price, `price_m2`. Time the cell with `%%time`, on its first line. The cell after the solution checks that `price_m2` equals `naive`, and shows the most expensive towns among those with at least 100 sales.

You should find a computation tens to hundreds of times faster, the same prices, and Paris arrondissements at the top.

In [ ]:
# Your code here

#### Additional hints

- A named aggregation takes one (column, function) pair per column of the result: `agg(value=("valeur_fonciere", "sum"), ...)`.
- `"size"` counts each group's rows.
- `"sum"` is a built-in function, in C: no group is ever handed to Python.

A skeleton, where each `...` is yours to fill in:

```python
%%time
totals = towns.agg(
  value=("valeur_fonciere", "sum"),
  area=...,
  sales=...,
)
price_m2 = ...
```

#### Solution

In [ ]:
%%time
totals = towns.agg(
  value=("valeur_fonciere", "sum"),
  area=("surface_reelle_bati_total", "sum"),
  sales=("valeur_fonciere", "size"),
)
price_m2 = totals["value"] / totals["area"]

In [ ]:
print("Same prices as the naive version:", np.allclose(price_m2, naive))
price_m2[totals["sales"] >= 100].nlargest(10).round()

### 3.2 `query` and `eval` (optional)

The sales of apartments alone (with no house) are filtered by a mask, in the cell below, which also times this filter and the price per m² of these sales.

Write the same filter with `dvf.query(...)`, in an `apartments_query()` function, and the price per m² of `apartments` with `apartments.eval(...)`, then time them with `%timeit`: compare with the two timings of the cell below. Print the number of sales each filter keeps, and the median price per m² of an apartment.

You should find 159,376 sales with both filters, `query` about twice as slow as the mask and `eval` about ten times slower than the division: their point is readability, not speed.

**Useful documentation**

- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.eval.html

In [ ]:
def apartments_mask():
  return dvf[
    (dvf["nombre_appartements"] > 0)
    & (dvf["nombre_maisons"] == 0)
    & (dvf["valeur_fonciere"] > 0)
    & (dvf["surface_reelle_bati_total"] > 0)
  ]


apartments = apartments_mask()
%timeit apartments_mask()
%timeit apartments["valeur_fonciere"] / apartments["surface_reelle_bati_total"]

In [ ]:
# Your code here

#### Additional hints

- In a query's text, columns are named without `dvf[...]`, and `and` replaces `&`; two strings side by side, on two lines, make a single one.
- `eval` takes an expression on the columns, and returns a `Series`.

A skeleton, where each `...` is yours to fill in:

```python
def apartments_query():
  return dvf.query("nombre_appartements > 0 and nombre_maisons == 0 and ...")


%timeit apartments_query()
%timeit apartments.eval(...)
print(len(apartments_query()), len(apartments), "sales")
price = apartments.eval(...)
print(f"Median price of an apartment: {price.median():.0f} €/m²")
```

#### Solution

In [ ]:
def apartments_query():
  return dvf.query(
    "nombre_appartements > 0 and nombre_maisons == 0"
    " and valeur_fonciere > 0 and surface_reelle_bati_total > 0"
  )


%timeit apartments_query()
%timeit apartments.eval("valeur_fonciere / surface_reelle_bati_total")
print(len(apartments_query()), len(apartments), "sales")
price = apartments.eval("valeur_fonciere / surface_reelle_bati_total")
print(f"Median price of an apartment: {price.median():.0f} €/m²")

## 4. Compiling with Numba

How far is each sale from the nearest of France's ten largest cities? For each of the 508,837 sales, it takes its ten distances to the cities, then the smallest one. In NumPy, broadcasting a `(n, 1)` column against a `(10,)` row computes each sale's ten distances at once, in a temporary `(n, 10)` array: that's the reference, timed below.

`haversine_njit` is your `haversine` compiled by Numba (`njit(haversine)`), to be called from another Numba function.

In [ ]:
CITIES = pd.DataFrame(
  [
    ("Paris", 48.8566, 2.3522),
    ("Marseille", 43.2965, 5.3698),
    ("Lyon", 45.7640, 4.8357),
    ("Toulouse", 43.6047, 1.4442),
    ("Nice", 43.7102, 7.2620),
    ("Nantes", 47.2184, -1.5536),
    ("Montpellier", 43.6108, 3.8767),
    ("Strasbourg", 48.5734, 7.7521),
    ("Bordeaux", 44.8378, -0.5792),
    ("Lille", 50.6292, 3.0573),
  ],
  columns=["city", "latitude", "longitude"],
)
lat, lon = dvf["latitude"].to_numpy(), dvf["longitude"].to_numpy()
lat_c, lon_c = CITIES["latitude"].to_numpy(), CITIES["longitude"].to_numpy()
haversine_njit = njit(haversine)


def nearest_city_distance_numpy(lat, lon, lat_c, lon_c):
  distances = haversine(lat[:, np.newaxis], lon[:, np.newaxis], lat_c, lon_c)
  return distances.min(axis=1)  # the smallest of each sale's 10 distances


numba_times = {
  "NumPy": measure(lambda: nearest_city_distance_numpy(lat, lon, lat_c, lon_c))
}
numba_times

### 4.1 `@njit`, then `parallel=True`

Numba compiles a Python function that loops over NumPy arrays. Write `nearest_city_distance(lat, lon, lat_c, lon_c)`, decorated with `@njit`, with two loops: for each sale `i`, the smallest of the distances `haversine_njit(lat[i], lon[i], lat_c[j], lon_c[j])` to the cities `j`, with no temporary array.

Then write `nearest_city_distance_parallel`, the same function decorated with `@njit(parallel=True)`, whose loop over the sales goes from `range` to `prange`: its iterations, independent, are spread over the cores.

Time the first call of each one with `%time`: it compiles the function. Then add their times, `measure(lambda: ...)`, to `numba_times`, under the names `"@njit"` and `"@njit(parallel=True)"`. The cell after the solution checks their results and shows the timings.

You should find a compilation of about a second, `@njit` a bit faster than NumPy, which fills then reads its temporary array, and `parallel=True` up to twice as fast again on 2 cores.

In [ ]:
@njit
def nearest_city_distance(lat, lon, lat_c, lon_c): ...  # Your code here


@njit(parallel=True)
def nearest_city_distance_parallel(lat, lon, lat_c, lon_c): ...  # Your code here

#### Additional hints

- `np.empty(len(lat))` allocates the result's array; `np.inf` is a good starting point for a smallest value, which `min(a, b)` updates.
- In an `@njit` function, a loop over numbers is compiled, as fast as in C: that's what makes the `(n, 10)` array unnecessary.
- `prange` comes from `numba`, imported at the start of the notebook; only the outer loop becomes `prange`.
- A Numba `TypingError` often comes from an argument that isn't a NumPy array (a `Series`, for example): pass `lat`, `lon`, `lat_c` and `lon_c`.
- `%time nearest = ...`: with no assignment, `%time` would print the whole array.

A skeleton, where each `...` is yours to fill in:

```python
@njit
def nearest_city_distance(lat, lon, lat_c, lon_c):
  nearest = np.empty(len(lat))
  for i in range(len(lat)):
    smallest = np.inf
    for j in range(len(lat_c)):
      smallest = min(smallest, ...)
    nearest[i] = smallest
  return nearest


@njit(parallel=True)
def nearest_city_distance_parallel(lat, lon, lat_c, lon_c): ...


%time nearest = nearest_city_distance(lat, lon, lat_c, lon_c)
%time nearest = nearest_city_distance_parallel(lat, lon, lat_c, lon_c)
numba_times["@njit"] = measure(lambda: nearest_city_distance(lat, lon, lat_c, lon_c))
numba_times["@njit(parallel=True)"] = ...
```

#### Solution

In [ ]:
@njit
def nearest_city_distance(lat, lon, lat_c, lon_c):
  nearest = np.empty(len(lat))
  for i in range(len(lat)):
    smallest = np.inf
    for j in range(len(lat_c)):
      smallest = min(smallest, haversine_njit(lat[i], lon[i], lat_c[j], lon_c[j]))
    nearest[i] = smallest
  return nearest


@njit(parallel=True)
def nearest_city_distance_parallel(lat, lon, lat_c, lon_c):
  nearest = np.empty(len(lat))
  for i in prange(len(lat)):  # independent iterations, spread over the cores
    smallest = np.inf
    for j in range(len(lat_c)):
      smallest = min(smallest, haversine_njit(lat[i], lon[i], lat_c[j], lon_c[j]))
    nearest[i] = smallest
  return nearest


%time nearest = nearest_city_distance(lat, lon, lat_c, lon_c)
%time nearest = nearest_city_distance_parallel(lat, lon, lat_c, lon_c)
numba_times["@njit"] = measure(lambda: nearest_city_distance(lat, lon, lat_c, lon_c))
numba_times["@njit(parallel=True)"] = measure(
  lambda: nearest_city_distance_parallel(lat, lon, lat_c, lon_c)
)

In [ ]:
nearest = nearest_city_distance_parallel(lat, lon, lat_c, lon_c)
numpy_reference = nearest_city_distance_numpy(lat, lon, lat_c, lon_c)
print("Same distances as with NumPy:", np.allclose(nearest, numpy_reference))
print(
  f"{(nearest < 20).mean():.0%} of the sales less than 20 km from one of the ten cities"
)
time_table(numba_times)

### 4.2 A loop that doesn't vectorize: a reference price

Numba mostly serves the loops where each step depends on the previous one, which no operation on whole columns replaces. The cell below gives the price per m² of the 2022 Paris sales, in the order of their dates (`paris_price`), and its trend, the rolling median over 500 sales, as a NumPy array (`trend`).

A reference price follows this trend, but only changes when the trend moves away from it by more than a threshold: it starts at the first value, and takes the value `price[i]` when the gap between `price[i]` and the reference exceeds `threshold` times the reference. Write `reference_price(price, threshold)`, a Python function with no decorator, which returns the array of the references, as long as `price`. Compile it with `reference_price_njit = njit(reference_price)`, as for `haversine_njit`, and time its first call on `trend`, with a 2% threshold, with `%time`. Then store the times of both versions in `reference_times`, under the names `"Python"` and `"@njit"`. The cell after the solution checks that they give the same references, and counts the changes.

You should find a reference that only changes about twenty times over nearly 17,500 sales, and the compiled version about a hundred times faster than Python.

In [ ]:
paris = built[built["code_departement"] == "75"].sort_values(
  ["mois_mutation", "jour_mutation"], ignore_index=True
)
paris_price = paris["valeur_fonciere"] / paris["surface_reelle_bati_total"]
trend = paris_price.rolling(500).median().dropna().to_numpy()
len(paris_price), len(trend)

In [ ]:
def reference_price(price, threshold):
  """The reference that follows `price` when it moves by more than `threshold`."""
  ...  # Your code here

#### Additional hints

- A loop over `range(len(price))` keeps the current reference in a variable, which an `if` updates, and writes it at each step into an array reserved by `np.empty(len(price))`.
- `abs(a - b) > threshold * b` compares a relative gap with no division.
- An example to check: `reference_price(np.array([10000.0, 10100, 10300, 10250, 9900]), 0.02)` gives `[10000, 10000, 10300, 10300, 9900]`: 10,100 is 1% away from 10,000, 10,300 3%, 10,250 less than 1% away from 10,300, and 9,900 nearly 4%.
- Numba compiles for NumPy arrays: `trend` is one, `paris_price` (a `Series`) isn't.
- `%time reference = ...`: with no assignment, `%time` would print the whole array.

A skeleton, where each `...` is yours to fill in:

```python
def reference_price(price, threshold):
  reference = np.empty(len(price))
  current = price[0]
  for i in range(len(price)):
    if ...:
      current = price[i]
    reference[i] = ...
  return reference


reference_price_njit = njit(reference_price)
%time reference = reference_price_njit(trend, 0.02)
reference_times = {}
reference_times["Python"] = measure(lambda: reference_price(trend, 0.02))
reference_times["@njit"] = ...
```

#### Solution

In [ ]:
def reference_price(price, threshold):
  """The reference that follows `price` when it moves by more than `threshold`."""
  reference = np.empty(len(price))
  current = price[0]
  for i in range(len(price)):
    if abs(price[i] - current) > threshold * current:
      current = price[i]  # each step depends on the previous one
    reference[i] = current
  return reference


reference_price_njit = njit(reference_price)
%time reference = reference_price_njit(trend, 0.02)
reference_times = {}
reference_times["Python"] = measure(lambda: reference_price(trend, 0.02))
reference_times["@njit"] = measure(lambda: reference_price_njit(trend, 0.02))

In [ ]:
reference = reference_price_njit(trend, 0.02)
equal = np.array_equal(reference, reference_price(trend, 0.02))
print("Same references with and without @njit:", equal)
print(f"{(np.diff(reference) != 0).sum()} changes over {len(reference)} sales")
time_table(reference_times)

### 4.3 `engine="numba"` in pandas (optional)

With `engine="numba"`, pandas compiles a function with Numba itself: in `rolling(...).apply(f, raw=True)`, which calls `f` on each window, a NumPy array, and in a `groupby`'s `agg`. Time it in these two cases, against the built-in method that does the same computation:

- the rolling median of `paris_price` over 500 sales: `rolling(500).apply(np.median, raw=True, engine="numba")` against `rolling(500).median()`, into `rolling_times`;
- the interquartile range (IQR) of each town's price per m², `np.percentile(x, 75) - np.percentile(x, 25)` over its sales' prices (`town_prices`, cell below): `agg(group_iqr, engine="numba")`, where `group_iqr(values, index)` has the signature pandas requires, against `quantile(0.75) - quantile(0.25)`, into `iqr_times`.

Store each time under the name `"Numba"` or under the built-in method's, and time the first call of each Numba version with `%time`: it compiles the function. The cell after the solution checks the results and shows the timings.

You should find `median()` about ten times faster than Numba, but Numba faster than the two `quantile`, which sort each group twice: a built-in method isn't always the fastest, only measuring tells them apart.

In [ ]:
built_price = built["valeur_fonciere"] / built["surface_reelle_bati_total"]
town_prices = built_price.groupby(
  [built["code_departement"], built["nom_commune"]], observed=True
)

In [ ]:
rolling_times = {}
iqr_times = {}
# Your code here

#### Additional hints

- `raw=True` passes each window as a NumPy array, which `engine="numba"` requires: Numba can't compile a `Series`.
- With `engine="numba"`, pandas passes the group's values and its index, two NumPy arrays, to `agg`'s function: `def group_iqr(values, index)`, even if it doesn't use `index`.
- `engine="numba"` raises a `TypingError` when the `Series`' index is made of strings: that of `built_price` comes from part 1's `RangeIndex` (`reset_index(drop=True)`).
- The first call compiles; the next ones reuse the compiled code, which pandas caches.

A skeleton, where each `...` is yours to fill in:

```python
windows = paris_price.rolling(500)
%time median = windows.apply(np.median, raw=True, engine="numba")
rolling_times = {}
rolling_times["Numba"] = measure(lambda: ...)
rolling_times["median()"] = measure(lambda: windows.median())


def group_iqr(values, index):
  return ...


%time iqr = town_prices.agg(group_iqr, engine="numba")
iqr_times = {}
iqr_times["Numba"] = ...
iqr_times["quantile"] = measure(lambda: ...)
```

#### Solution

In [ ]:
windows = paris_price.rolling(500)
%time median = windows.apply(np.median, raw=True, engine="numba")
rolling_times = {}
rolling_times["Numba"] = measure(
  lambda: windows.apply(np.median, raw=True, engine="numba")
)
rolling_times["median()"] = measure(lambda: windows.median())


def group_iqr(values, index):
  return np.percentile(values, 75) - np.percentile(values, 25)


%time iqr = town_prices.agg(group_iqr, engine="numba")
iqr_times = {}
iqr_times["Numba"] = measure(lambda: town_prices.agg(group_iqr, engine="numba"))
iqr_times["quantile"] = measure(
  lambda: town_prices.quantile(0.75) - town_prices.quantile(0.25)
)

In [ ]:
median = paris_price.rolling(500).median()
numba_median = paris_price.rolling(500).apply(np.median, raw=True, engine="numba")
print("Same medians:", np.allclose(median, numba_median, equal_nan=True))
iqr = town_prices.agg(group_iqr, engine="numba")
quantile_iqr = town_prices.quantile(0.75) - town_prices.quantile(0.25)
print("Same IQRs:", np.allclose(iqr, quantile_iqr))
pd.concat(
  {
    "rolling median": time_table(rolling_times),
    "IQR by town": time_table(iqr_times),
  }
)

## 5. Scaling up: 2014–2022

Each department's median price per m², over the nine years: 10.8 million sales. pandas reads the three useful columns (`PRICE_COLUMNS`), and the `id_mutation` index with them, then `median_prices` filters and groups: ten seconds or more, and more than 2 GB of memory at the peak. `by_department` puts a result in a common shape (a `Series` indexed by the department's code, as text, sorted), to compare those of two libraries.

In [ ]:
PRICE_COLUMNS = ["code_departement", "valeur_fonciere", "surface_reelle_bati_total"]


def median_prices(dvf):
  """The median price per m² of the buildings of each department of `dvf`."""
  built = dvf.query("surface_reelle_bati_total > 0")
  price = built["valeur_fonciere"] / built["surface_reelle_bati_total"]
  return price.groupby(built["code_departement"], observed=True).median()


def department_prices_pandas():
  return median_prices(pd.read_parquet(PATH, columns=PRICE_COLUMNS))


def by_department(price):
  """`price` as a Series indexed by the department's code, as text, sorted."""
  return pd.Series(price.to_numpy(), index=price.index.astype(str)).sort_index()


scale_times = {}
pandas_price, scale_times["pandas"] = stopwatch(department_prices_pandas)
print(f"pandas: {scale_times['pandas']:.1f} s")
pandas_price.nlargest(5).round()

### 5.1 Where does Modin lose its time?

Modin offers pandas' API on several cores: you only import `modin.pandas` instead of `pandas`. The cell below does the same computation with Modin: the same `median_prices` function, on a table read by `mpd.read_parquet`, and a result brought back to pandas by `.modin.to_pandas()`. Modin first starts its processes (Dask engine), and warns that it reads this partitioned Parquet dataset with pandas ("Defaulting to pandas") before spreading the table: that's expected.

In [ ]:
import modin.pandas as mpd


def department_prices_modin():
  dvf = mpd.read_parquet(PATH, columns=PRICE_COLUMNS)
  return median_prices(dvf).modin.to_pandas()


modin_price, scale_times["Modin"] = stopwatch(department_prices_modin)
equal = np.allclose(by_department(modin_price), by_department(pandas_price))
print("Same result as pandas:", equal)
time_table(scale_times)

Modin is slower than pandas on 2 cores. To find out where it loses its time, time separately, with `stopwatch`, the three steps of `department_prices_modin`: the reading (`mpd.read_parquet`), the computation (`median_prices`) and the conversion (`.modin.to_pandas()`), then the first two in pandas. Store the times in `steps`, one dictionary per library, under the names `"reading"`, `"computation"` and `"conversion"`; the cell after the solution shows them as a table.

You should find a reading more than twice as slow with Modin, which reads the Parquet dataset with pandas, then sends the table to its processes: that's where it loses its time. The computation, spread over the 2 cores, doesn't make up for it, and the conversion is negligible, the result having only about a hundred rows.

**Useful documentation**

- https://modin.readthedocs.io/en/stable/flow/modin/experimental/pandas.html#modin.pandas.DataFrame.modin.to_pandas

In [ ]:
steps = {"pandas": {}, "Modin": {}}
# Your code here

#### Additional hints

- `stopwatch` takes a function with no argument: `stopwatch(lambda: pd.read_parquet(PATH, columns=PRICE_COLUMNS))` returns the table and its time.
- Keep each step's result for the next one: the table read for `median_prices`, its medians for `.modin.to_pandas()`.
- The pandas table takes nearly a gigabyte: `del pandas_dvf`, once its computation is timed, frees it before Modin's reading.

A skeleton, where each `...` is yours to fill in:

```python
steps = {"pandas": {}, "Modin": {}}
pandas_dvf, steps["pandas"]["reading"] = stopwatch(
  lambda: pd.read_parquet(PATH, columns=PRICE_COLUMNS)
)
_, steps["pandas"]["computation"] = stopwatch(lambda: median_prices(pandas_dvf))
del pandas_dvf
modin_dvf, steps["Modin"]["reading"] = ...
medians, steps["Modin"]["computation"] = ...
_, steps["Modin"]["conversion"] = ...
```

#### Solution

In [ ]:
steps = {"pandas": {}, "Modin": {}}
pandas_dvf, steps["pandas"]["reading"] = stopwatch(
  lambda: pd.read_parquet(PATH, columns=PRICE_COLUMNS)
)
_, steps["pandas"]["computation"] = stopwatch(lambda: median_prices(pandas_dvf))
del pandas_dvf  # frees its memory before Modin's reading
modin_dvf, steps["Modin"]["reading"] = stopwatch(
  lambda: mpd.read_parquet(PATH, columns=PRICE_COLUMNS)
)
medians, steps["Modin"]["computation"] = stopwatch(lambda: median_prices(modin_dvf))
_, steps["Modin"]["conversion"] = stopwatch(lambda: medians.modin.to_pandas())
del modin_dvf

In [ ]:
pd.DataFrame(steps).round(2)

### 5.2 Polars: Paris apartments, year by year

Another question, over the nine years: the median price per m² of Paris apartments (the sales of apartments alone, with no house, with a positive value and area, as in 3.2), year by year. The cell below answers it in pandas, reading only the Paris sales (`filters=`), and times the computation into `paris_times`.

Write `paris_by_year_polars()`, the same computation in Polars: a lazy query on the Parquet folder, `pl.scan_parquet(PATH)`, then `filter`, `group_by` of the year (`annee_mutation`), `agg` of the median price per m², named `price_m2` (`alias`), `sort` by year and `collect()`, which runs it. Return a pandas `Series` of it, indexed by the year (`.to_pandas()`, `set_index`). Time it with `stopwatch`, into `by_year_polars` and `paris_times["Polars"]`.

You should find Polars faster than pandas, which also reads only the Paris sales, though, and a median price that went from about €8,000 per m² in 2014 to more than €10,600 from 2020 to 2022.

In [ ]:
def paris_by_year_pandas():
  dvf = pd.read_parquet(
    PATH,
    columns=[
      "annee_mutation",
      "valeur_fonciere",
      "surface_reelle_bati_total",
      "nombre_maisons",
      "nombre_appartements",
    ],
    filters=[("code_departement", "==", "75")],
  )
  apartments = dvf.query(
    "nombre_appartements > 0 and nombre_maisons == 0"
    " and valeur_fonciere > 0 and surface_reelle_bati_total > 0"
  )
  price = apartments["valeur_fonciere"] / apartments["surface_reelle_bati_total"]
  return price.groupby(apartments["annee_mutation"], observed=True).median()


paris_times = {}
by_year_pandas, paris_times["pandas"] = stopwatch(paris_by_year_pandas)
print(f"pandas: {paris_times['pandas']:.1f} s")
by_year_pandas.round()

In [ ]:
# Your code here

#### Additional hints

- `pl.scan_parquet(PATH)`, on the folder: Polars recognizes its `annee_mutation=2014`… subfolders and makes them the `annee_mutation` column; with the `f"{PATH}/**/*.parquet"` pattern, it doesn't create it.
- `pl.col("name")` stands for a column in an expression; expressions combine like pandas columns (`/`, `>`, `&`, a condition between parentheses) and have their methods (`.median()`, `.alias("name")`).
- Nothing is read before `collect()`: Polars first optimizes the whole plan.
- A Polars DataFrame has no index: `.to_pandas().set_index("annee_mutation")["price_m2"]`.

A skeleton, where each `...` is yours to fill in:

```python
def paris_by_year_polars():
  price = (
    pl.scan_parquet(PATH)
    .filter(
      (pl.col("code_departement") == "75")
      & (pl.col("nombre_appartements") > 0)
      & (pl.col("nombre_maisons") == 0)
      & ...
    )
    .group_by("annee_mutation")
    .agg(...)
    .sort("annee_mutation")
    .collect()
  )
  return ...


by_year_polars, paris_times["Polars"] = stopwatch(paris_by_year_polars)
print(f"Polars: {paris_times['Polars']:.1f} s")
```

#### Solution

In [ ]:
def paris_by_year_polars():
  price = (
    pl.scan_parquet(PATH)
    .filter(
      (pl.col("code_departement") == "75")
      & (pl.col("nombre_appartements") > 0)
      & (pl.col("nombre_maisons") == 0)
      & (pl.col("valeur_fonciere") > 0)
      & (pl.col("surface_reelle_bati_total") > 0)
    )
    .group_by("annee_mutation")
    .agg(
      (pl.col("valeur_fonciere") / pl.col("surface_reelle_bati_total"))
      .median()
      .alias("price_m2")
    )
    .sort("annee_mutation")
    .collect()
  )
  return price.to_pandas().set_index("annee_mutation")["price_m2"]


by_year_polars, paris_times["Polars"] = stopwatch(paris_by_year_polars)
print(f"Polars: {paris_times['Polars']:.1f} s")

### 5.3 DuckDB

Write `paris_by_year_duckdb()`, the same computation in SQL: `duckdb.sql(...)` on the Parquet files (`FROM '...'`, with the `**/*.parquet` pattern: DuckDB, for its part, recognizes the year folders and adds the `annee_mutation` column), `median(...)` of the price per m², `WHERE`, `GROUP BY`, `ORDER BY`, then `.df()` for a pandas DataFrame, of which you return a `Series` indexed by the year. Time it with `stopwatch`, into `by_year_duckdb` and `paris_times["DuckDB"]`. The cell after the solution compares the three libraries' results and timings.

You should find the same prices as in pandas and Polars, and DuckDB faster still than Polars.

In [ ]:
# Your code here

#### Additional hints

- An f-string inserts `PATH` into the query; in the query, the path is between single quotes, as is the department's code (`'75'`).
- `AND` combines the `WHERE` conditions; `AS price_m2` names the result's column.

A skeleton, where each `...` is yours to fill in:

```python
def paris_by_year_duckdb():
  price = duckdb.sql(f"""
    SELECT annee_mutation,
           median(...) AS price_m2
    FROM '{PATH}/**/*.parquet'
    WHERE code_departement = '75'
      AND ...
    GROUP BY annee_mutation
    ORDER BY annee_mutation
  """).df()
  return ...


by_year_duckdb, paris_times["DuckDB"] = stopwatch(paris_by_year_duckdb)
print(f"DuckDB: {paris_times['DuckDB']:.1f} s")
```

#### Solution

In [ ]:
def paris_by_year_duckdb():
  price = duckdb.sql(f"""
    SELECT annee_mutation,
           median(valeur_fonciere / surface_reelle_bati_total) AS price_m2
    FROM '{PATH}/**/*.parquet'
    WHERE code_departement = '75'
      AND nombre_appartements > 0 AND nombre_maisons = 0
      AND valeur_fonciere > 0 AND surface_reelle_bati_total > 0
    GROUP BY annee_mutation
    ORDER BY annee_mutation
  """).df()
  return price.set_index("annee_mutation")["price_m2"]


by_year_duckdb, paris_times["DuckDB"] = stopwatch(paris_by_year_duckdb)
print(f"DuckDB: {paris_times['DuckDB']:.1f} s")

In [ ]:
for name, by_year in [("Polars", by_year_polars), ("DuckDB", by_year_duckdb)]:
  equal = np.allclose(by_year.to_numpy(), by_year_pandas.to_numpy())
  print(f"{name} gives the same prices as pandas: {equal}")
time_table(paris_times)

## Key takeaways

- Measure first: `%timeit`, `%time`, `memory_usage(deep=True)`.
- Read less (`columns=`, `filters=`) and choose dtypes (`category`, `downcast`), column by column with `df.items()`: often the first gain, and the simplest.
- Vectorize: an operation on whole columns rather than a loop, `apply` or `np.vectorize`, which call Python for every row; `np.select` rather than a function with `if`.
- Group with built-in functions (`agg` of `"sum"`, `"size"`…), not with `apply`; `query` and `eval` serve readability.
- Numba, when a loop doesn't vectorize, or badly: `@njit`, `prange`, `engine="numba"`, and a measurement against the built-in method.
- Beyond memory, or to go faster: Polars and DuckDB rather than Modin, on a machine with few cores.